# Cox survival analysis

* `mydir`: path to the input data.
* `ostime_column`: survival duration column; other endpoints may be used instead of overall survival.
* `os`: event indicator; other survival endpoints may be used instead of overall survival.

In [ ]:
from lifelines import CoxPHFitter
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df
from sklearn.model_selection import train_test_split
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import fillna
from onekey_algo.custom.components.metrics import calc_value_95ci

def get_prediction(mn):
    subsets = get_param_in_cwd('subsets')
    prediction = pd.concat([pd.read_csv(f'results/{mn}_cox_predictions_{subset}.csv') for subset in subsets],
                           axis=0)
    prediction.columns = ['ID', mn, f'{mn}_Exp']
    # prediction[mn] = prediction[mn].map(lambda x: 10000 if x > 10000 else x)
    return prediction[['ID', f'{mn}']]

data = None
for mn in get_param_in_cwd('summary_models'):
    pred = get_prediction(mn)
    pred['ID'] = pred['ID'].astype(str)
    if data is None:
        data = pred
    else:
        data = pd.merge(data, pred, on='ID', how='left')
# data = normalize_df(data, not_norm=['ID', 'group'])
label_data = pd.read_csv(get_param_in_cwd('survival_file'))
# label_data = pd.merge(label_data, pd.read_csv('group.csv'), on='ID', how='inner')
# label_data = fillna(label_data, fill_mod='50%')
# data = pd.merge(data, label_data[['ID']], on='ID', how='inner').drop_duplicates('ID')
data = pd.merge(data, pd.read_csv(get_param_in_cwd('label_file')), on='ID', how='left')
data.to_csv('results/joinit_info.csv', index=False)
data

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.utils import concordance_index
from onekey_algo.custom.components.metrics import calc_value_95ci

import numpy as np

metrics = []
def get_ci(x, sample_num):
    l, h = calc_value_95ci(x, sample_num=sample_num)
    return f"{x:.3f} ({l:.3f}-{h:.3f})"

for mn in get_param_in_cwd('summary_models'):
    metric = []
    for subset in get_param_in_cwd('subsets'):
        subdata = data[data['group'] == subset]
#         display(subdata)
        metric.append(get_ci(concordance_index(subdata[get_param_in_cwd('duration_col')],
                                               -subdata[mn], 
                                               subdata[get_param_in_cwd('event_col')]), 
                             subdata.shape[0]))
    metrics.append(metric)
metrics = pd.DataFrame(np.array(metrics).T, columns=get_param_in_cwd('summary_models'))
metrics['Cohort'] = get_param_in_cwd('subsets')
metrics